# 원본 세션 데이터 → guidellm JSONL 변환

다중 세션 JSON(OpenAI 형식)을 로드해, 패치된 guidellm이 사용하는 JSONL 형식으로 저장합니다.

- 각 행: `{"messages": [...], "output_seq_len": N}`
- `messages`: 턴별 대화 메시지 배열(role/content)
- `output_seq_len`: 해당 턴의 생성 목표 길이(토큰 수, max_tokens 또는 기본값). guidellm이 `output_tokens_count_column`으로 인식

In [11]:
import json
from pathlib import Path

# notebook을 data/ 디렉터리에서 실행하면 상대 경로가 data/ 기준이 됨
DATA_DIR = Path(".")

# 원본 세션 JSON 경로 (data/ 내 파일 또는 절대 경로)
INPUT_JSON = DATA_DIR / "e22_sessions_openai.json"

# 저장할 guidellm JSONL 경로
OUTPUT_JSONL = DATA_DIR / "e22_sessions_guidellm_long.jsonl"

# concurrent_session_test.py 기준 (--num-sessions 22 --skip-first-turns 40 --max-turns 40)
NUM_SESSIONS = 22
SKIP_FIRST_TURNS = 120     # 세션당 앞 N턴 건너뛰기
MAX_TURNS = 40            # 건너뛴 뒤 세션당 최대 턴 수 (즉, 41~80번째 턴만 사용)


In [ ]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

sessions = data.get("sessions", [])
print(f"로드: {INPUT_JSON}")
print(f"  세션 수: {len(sessions)}")
print(f"  총 턴 수: {sum(len(s.get('turns', [])) for s in sessions)}")


In [12]:
def _normalize_content(c):
    """content를 항상 리스트로 통일 (PyArrow 스키마 오류 방지)."""
    if isinstance(c, list):
        return c
    if isinstance(c, str):
        return [{"type": "text", "text": c}]
    return [{"type": "text", "text": str(c)}]


def _normalize_messages(msgs):
    out = []
    for m in msgs:
        m = dict(m)
        m["content"] = _normalize_content(m.get("content"))
        out.append(m)
    return out


def sessions_to_guidellm_jsonl(
    sessions: list,
    output_path: Path,
    max_turns: int | None = None,
    skip_first_turns: int = 0,
    default_output_seq_len: int = 256,
) -> int:
    """세션 리스트를 guidellm용 JSONL로 저장. 반환: 저장된 행 수."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for session in sessions:
        turns = session.get("turns", [])
        if skip_first_turns > 0:
            turns = turns[skip_first_turns:]
        if max_turns is not None:
            turns = turns[:max_turns]
        for turn in turns:
            messages = []
            if turn.get("system"):
                messages.append({"role": "system", "content": _normalize_content(turn["system"])})
            messages.extend(_normalize_messages(turn.get("messages", [])))
            if not messages:
                continue
            rows.append({
                "messages": messages,
                "output_seq_len": turn.get("output_seq_len") or turn.get("max_tokens") or default_output_seq_len,
            })
    with open(output_path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return len(rows)


sessions = data.get("sessions", [])
print(f"로드: {INPUT_JSON}")
print(f"  세션 수: {len(sessions)}")
print(f"  총 턴 수: {sum(len(s.get('turns', [])) for s in sessions)}")

n = sessions_to_guidellm_jsonl(
    sessions,
    OUTPUT_JSONL,
    max_turns=MAX_TURNS,
    skip_first_turns=SKIP_FIRST_TURNS,
)
print(f"저장: {OUTPUT_JSONL} ({n} 행)")


로드: e22_sessions_openai.json
  세션 수: 22
  총 턴 수: 17487
저장: e22_sessions_guidellm_long.jsonl (880 행)


In [13]:
# 저장된 JSONL 미리보기 (첫 2행)
with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 2:
            break
        row = json.loads(line)
        msg_count = len(row.get("messages", []))
        out_len = row.get("output_seq_len", 0)
        print(f"행 {i+1}: messages={msg_count}개, output_seq_len={out_len}")


행 1: messages=2개, output_seq_len=32000
행 2: messages=2개, output_seq_len=32000


In [44]:

import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o")

skip_first_turns = 120     # 세션당 앞 N턴 건너뛰기
max_turns = 40

rows = []
for session in sessions:
    turns = session.get("turns", [])
    if skip_first_turns > 0:
        turns = turns[skip_first_turns:]
    if max_turns is not None:
        turns = turns[:max_turns]
    for turn in turns:
        messages = []
        if turn.get("system"):
            messages.append({"role": "system", "content": _normalize_content(turn["system"])})
        messages.extend(_normalize_messages(turn.get("messages", [])))
        if not messages:
            continue

        # if len(enc.encode('\n'.join([m['content'][0]['text'] for m in messages]))) > 10000:
        rows.append({
            "messages": messages,
            "output_seq_len": turn.get("output_seq_len") or turn.get("max_tokens"),
        })

len_chars = []
for r in rows:
    len_chars.append(len(enc.encode('\n'.join([m['content'][0]['text'] for m in r['messages']]))))

print(min(len_chars))
print(max(len_chars))
print(mean(len_chars))


268
21851
4004.552272727273


In [ ]:

skip_first_turns = 40     # 세션당 앞 N턴 건너뛰기
max_turns = 40

rows = []
for session in sessions:
    turns = session.get("turns", [])
    if skip_first_turns > 0:
        turns = turns[skip_first_turns:]
    if max_turns is not None:
        turns = turns[:max_turns]
    for turn in turns:
        messages = []
        if turn.get("system"):
            messages.append({"role": "system", "content": _normalize_content(turn["system"])})
        messages.extend(_normalize_messages(turn.get("messages", [])))
        if not messages:
            continue
        rows.append({
            "messages": messages,
            "output_seq_len": turn.get("output_seq_len") or turn.get("max_tokens"),
        })

len_chars = []
for r in rows:
    len_chars.append(len(enc.encode('\n'.join([m['content'][0]['text'] for m in r['messages']]))))

print(min(len_chars))
print(max(len_chars))
print(mean(len_chars))


274
8798
2242.135227272727
